# Delta-2 STEP 2: scaffold sanity-train

Reuses generate_delta() unchanged -> novelty-weighted pool -> fixed delta_vec -> embedding
anchor + swappable aux. NO token decoder (see delta_system/DELTA2_DESIGN.md). This is a SANITY
check: does the scaffold actually learn? We want anchor cos up, and either paraphrase
separation (edit/para norm >1.5) or NLI train acc above chance. Full battery is step 3.

In [ ]:
# Cell 1 - install + clone/pull + HF token (avoids unauthenticated rate-limit stalls)
!pip install transformers scikit-learn scipy datasets -q
import os, sys
from pathlib import Path

# HF token: add a NEW token in Kaggle (Add-ons -> Secrets, name it HF_TOKEN). Never paste it here.
try:
    from kaggle_secrets import UserSecretsClient
    _t = UserSecretsClient().get_secret("HF_TOKEN")
    os.environ["HF_TOKEN"] = _t; os.environ["HUGGING_FACE_HUB_TOKEN"] = _t
    print("HF token: loaded")
except Exception as e:
    print("HF token: NONE (downloads may be rate-limited) ->", e)

REPO = Path('/kaggle/working/multihop-retrieval')
if REPO.exists():
    print(os.popen(f'git -C {REPO} pull').read())
else:
    os.system(f'git clone https://github.com/haaaaaaayrewugrhyw/multihop-retrieval.git {REPO}')
sys.path.insert(0, str(REPO / 'delta_system'))
print('OK')

In [ ]:
# Cell 2 - sanity-train with the PARAPHRASE aux (surface-invariance)
# python -u = unbuffered, so prints stream live (no "stuck" illusion from buffering)
import subprocess, sys
r = subprocess.run([sys.executable, '-u',
    '/kaggle/working/multihop-retrieval/delta_system/delta2_model.py',
    '--aux', 'paraphrase', '--steps', '400'])
print('exit code', r.returncode)

In [ ]:
# Cell 3 - sanity-train with the NLI aux (delta carries the relation)
import subprocess, sys
r = subprocess.run([sys.executable, '-u',
    '/kaggle/working/multihop-retrieval/delta_system/delta2_model.py',
    '--aux', 'nli', '--steps', '400'])
print('exit code', r.returncode)